In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential,layers
from keras.layers import Dense,Flatten
from keras.applications import ResNet50V2
from keras.applications.vgg16 import VGG16
import matplotlib.pyplot as plt

In [2]:
import os
from datasets import load_dataset

# 1. Load the dataset
dataset = load_dataset("rajistics/indian_food_images", split="train[:5000]")

# 2. Get the class names (so we can name the folders correctly)
# The dataset stores these internally
class_names = dataset.features['label'].names

# 3. Save images to folders
print("Saving images to disk... this may take a minute.")
for i, item in enumerate(dataset):
    # Get the image and label name
    image = item['image']
    label_name = class_names[item['label']]

    # Create the folder if it doesn't exist (e.g., "indian_food/samosa")
    folder_path = f"indian_food/{label_name}"
    os.makedirs(folder_path, exist_ok=True)

    # Save the image
    # We convert to RGB to fix the "grayscale" crash you had earlier
    image.convert("RGB").save(f"{folder_path}/{i}.jpg")

print("Done! Data is ready for Keras.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/660 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-dbae6752a5d31c(…):   0%|          | 0.00/1.36G [00:00<?, ?B/s]

data/test-00000-of-00001-899c7c7e401d279(…):   0%|          | 0.00/241M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5328 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/941 [00:00<?, ? examples/s]

Saving images to disk... this may take a minute.
Done! Data is ready for Keras.


In [3]:
import tensorflow as tf

# Load Training Data
train_ds = tf.keras.utils.image_dataset_from_directory(
    "indian_food",
    validation_split=0.2,       # Automatically create a validation split
    subset="training",
    seed=123,
    image_size=(224, 224),      # Resizes everything for you!
    batch_size=32
)

# Load Validation Data
val_ds = tf.keras.utils.image_dataset_from_directory(
    "indian_food",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

# 5. Cache & Prefetch (Validation)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
# Train
# history = model.fit(train_ds, validation_data=val_ds, epochs=10)


Found 5000 files belonging to 20 classes.
Using 4000 files for training.
Found 5000 files belonging to 20 classes.
Using 1000 files for validation.


In [4]:

conv_base=ResNet50V2(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling=max,
    classes=20,
    classifier_activation="softmax",
    name="resnet50v2",
)
# include_top=False tells us to ignore the dense layer

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
conv_base.summary()

Model: "resnet50v2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, None,      │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, None,      │      9,472 │ conv1_pad[0][0]   │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, None,      │          0 │ conv1_conv[0][0]  │
│ (ZeroPadding2D)     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, None,      │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, None,      │        256 │ pool1_pool[0][0]  │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, None,      │          0 │ conv2_block1_pre… │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, None,      │      4,096 │ conv2_block1_pre… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, None,      │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, None,      │          0 │ conv2_block1_1_b… │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_pad  │ (None, None,      │          0 │ conv2_block1_1_r… │
│ (ZeroPadding2D)     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, None,      │     36,864 │ conv2_block1_2_p… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, None,      │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, None,      │          0 │ conv2_block1_2_b… │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, None,      │     16,640 │ conv2_block1_pre… │
│ (Conv2D)            │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, None,      │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_out    │ (None, None,      │          0 │ conv2_block1_0_c

 Total params: 23,564,800 (89.89 MB)

 Trainable params: 23,519,360 (89.72 MB)

 Non-trainable params: 45,440 (177.50 KB)

In [6]:

# 1. Define the Augmentation Block
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

# 2. Add it inside your model definition (as the very first step)
model = tf.keras.Sequential([
    # Input Layer
    tf.keras.Input(shape=(224, 224, 3)),

    # --- AUGMENTATION HAPPENS HERE ---
    data_augmentation,
    # ---------------------------------

    # Rescaling (0-255 -> 0-1) - CRITICAL for most models
    tf.keras.layers.Rescaling(1./255),

    # Your Base Model (CNN / ResNet / etc.)
    # ...
])
model.add(conv_base)
model.add(Flatten())
model.add(Dense(265,activation='relu'))
model.add(Dense(20,activation='softmax'))

In [7]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50v2 (Functional)         │ (None, 7, 7, 2048)     │    23,564,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 265)            │    26,593,545 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         5,320 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,163,665 (191.36 MB)

 Trainable params: 50,118,225 (191.19 MB)

 Non-trainable params: 45,440 (177.50 KB)

In [8]:
conv_base.trainable=True
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [9]:
model.compile(optimizer=optimizer,loss='SparseCategoricalCrossentropy',metrics=['accuracy'])

In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models


history = model.fit(
    train_ds,                  # Your tf.data.Dataset
    epochs=20,
    validation_data=val_ds
)

Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 125s 525ms/step - accuracy: 0.4158 - loss: 2.2231 - val_accuracy: 0.7000 - val_loss: 1.1110
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 60s 482ms/step - accuracy: 0.7744 - loss: 0.7531 - val_accuracy: 0.7720 - val_loss: 0.8753
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 59s 475ms/step - accuracy: 0.8597 - loss: 0.4600 - val_accuracy: 0.7830 - val_loss: 0.8969
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 59s 476ms/step - accuracy: 0.8976 - loss: 0.3391 - val_accuracy: 0.8010 - val_loss: 0.8050
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 60s 477ms/step - accuracy: 0.9196 - loss: 0.2530 - val_accuracy: 0.8160 - val_loss: 0.8751
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 59s 476ms/step - accuracy: 0.9438 - loss: 0.1812 - val_accuracy: 0.8010 - val_loss: 0.9695
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 61s 489ms/step - accuracy: 0.9419 - loss: 0.1832 - val_accuracy: 0.7790 - val_loss: 1.1751
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 60s 477ms/step - accuracy: 0.9446 - loss: 

First, we need to convert the PIL Image objects into numerical arrays (e.g., NumPy arrays). Then, we can apply Min-Max scaling to transform the pixel values from their original range (e.g., 0-255 for 8-bit images) to a desired range, typically [0, 1].

In [11]:
model.save('resnet_model.keras')

In [12]:
class_names=['burger','butter_naan','chai','chapati','chole_bhature','dal_makhani','dhokla','fried_rice','idli','jalebi','kaathi_rolls','kadai_paneer','kulfi','masala_dosa','momos','paani_puri','pakode','pav_bhaji','pizza','samosa']